# Regular Expressions and Corpus Analysis


Text processing on a real parallel corpus using regular expressions and standard library Python: counting tokens and types, searching for substrings across a vocabulary, building a frequency histogram, and breaking a Caesar cipher with frequency analysis.

Coursework for ICS472 (Natural Language Processing), KFUPM.


## The corpus


### The United Nations Corpus


The [United Nations Corpus](http://web.science.mq.edu.au/~rdale/publications/papers/2009/MTS-2009-Rafalovitch.pdf) is a six-language parallel corpus of UN General Assembly resolutions, in Arabic, Chinese, English, French, Russian and Spanish. It is described in:

> Alexandre Rafalovitch and Robert Dale. 2009. United Nations General Assembly Resolutions: A Six-Language Parallel Corpus. In *Proceedings of the MT Summit XII*, pages 292-299, Ottawa, Canada.

The source distribution is a single TMX file, `uncorpora_plain_20090831.tmx`, holding all six languages interleaved with XML markup. At roughly 156 MB it is too large to keep in this repository, so it is not committed here; the cell that scans it is the only one that needs it.

Everything after that works on the **English segments extracted from the TMX with the XML stripped**, saved as `uncorpus.eng.txt` and committed here gzipped as `uncorpus.eng.txt.gz`. The extraction was done with `grep`, and the result is 18,009,005 bytes, which `wc -c` confirms.


### Regular expressions


Each cell below notes the regular expression it uses as a comment, in [regex101](https://regex101.com/) notation.


## Analysis


A note on tokenization: unless a question says otherwise, a **word** here is any sequence of one or more non-space characters. Later questions switch to a letters-only definition, which is stated where it applies.


## How many lines mention human rights?

Case-insensitive, counting lines rather than occurrences, scanning the full six-language TMX.


In [1]:
# regex101: /human rights/i
import re

tmx_path = "uncorpora_plain_20090831.tmx"
human_rights_pattern = re.compile(r"human rights", re.IGNORECASE)

matching_line_count = 0
with open(tmx_path, "r", encoding="utf-8", errors="ignore") as tmx_file:
    for line in tmx_file:
        # Count lines (not occurrences) that contain the phrase
        if human_rights_pattern.search(line):
            matching_line_count += 1

print("{:,} lines of text.".format(matching_line_count))

5,664 lines of text.


## The English corpus

The remaining questions use `uncorpus.eng.txt`, the English side of the corpus with markup removed. A word here is any sequence of one or more non-space characters.


In [2]:
import re
import gzip

eng_txt_path = "uncorpus.eng.txt.gz"

with gzip.open(eng_txt_path, "rt", encoding="utf-8", errors="ignore") as eng_file:
    eng_text = eng_file.read()

# "word" = sequence of one or more non-space characters
tokens = eng_text.split()

**a.** Count the total number of words (tokens).

In [3]:
token_count = len(tokens)
print("Total tokens: {:,}".format(token_count))

Total tokens: 2,685,538


**b.** Count the total number of unique words (types).

In [4]:
unique_token_count = len(set(tokens))
print("Unique tokens (types): {:,}".format(unique_token_count))

Unique tokens (types): 37,032


**c.** Count the total number of unique words (types) ignoring capitalization.

In [5]:
lower_tokens = [token.lower() for token in tokens]
unique_lower_token_count = len(set(lower_tokens))
print("Unique tokens (types), case-insensitive: {:,}".format(unique_lower_token_count))

Unique tokens (types), case-insensitive: 33,364


**d.** Count the total number of words (tokens) made out of digits only (e.g., 9000).

In [6]:
digits_only_pattern = re.compile(r"^\d+$")

digits_only_count = sum(1 for token in tokens if digits_only_pattern.match(token))
print("Digits-only tokens: {:,}".format(digits_only_count))

Digits-only tokens: 35,857


**e.** Count the total number of words (tokens) made out of digits and at least one other non alphabetic character (e.g. 8,000.00 or 8-8).

In [7]:
has_digit_pattern = re.compile(r"\d")
has_letter_pattern = re.compile(r"[A-Za-z]")
has_non_alpha_non_digit_pattern = re.compile(r"[^A-Za-z0-9]")

digits_and_symbol_count = 0
for token in tokens:
    if has_digit_pattern.search(token) and not has_letter_pattern.search(token) and has_non_alpha_non_digit_pattern.search(token):
        digits_and_symbol_count += 1

print("Digits + non-alphabetic (no letters) tokens: {:,}".format(digits_and_symbol_count))

Digits + non-alphabetic (no letters) tokens: 71,337


## War and peace in the UN

Ignoring letter case and all non-alphabetic characters, so tokens are runs of letters only.


In [8]:
# regex101: /[A-Za-z]+/g
import re
import gzip

eng_txt_path = "uncorpus.eng.txt.gz"

with gzip.open(eng_txt_path, "rt", encoding="utf-8", errors="ignore") as eng_file:
    corpus_text = eng_file.read().lower()

# Letters-only tokenization
word_tokens = re.findall(r"[a-z]+", corpus_text)

unique_words = set(word_tokens)

**a.** What is the count of unique words containing the substring "war"?

In [9]:
# regex101: /war/
war_anywhere = {w for w in unique_words if "war" in w}
print("{:,}".format(len(war_anywhere)))

43


**b.** What is the count of unique words starting with the substring "war"?

In [10]:
# regex101: /^war/
war_prefix = {w for w in unique_words if w.startswith("war")}
print("{:,}".format(len(war_prefix)))

20


**c.** What is the count of unique words ending with the substring "war"?

In [11]:
# regex101: /war$/
war_suffix = {w for w in unique_words if w.endswith("war")}
print("{:,}".format(len(war_suffix)))

1


**d.** What is the count of unique words containing the substring "war" but neither start nor end with it?

In [12]:
# regex101: /war/  (excluding ^war and war$)
war_middle = {w for w in unique_words if ("war" in w and not w.startswith("war") and not w.endswith("war"))}
print("{:,}".format(len(war_middle)))

23


**e.** What is the count of unique words containing the substring "peace"?

In [13]:
# regex101: /peace/
peace_anywhere = {w for w in unique_words if "peace" in w}
print("{:,}".format(len(peace_anywhere)))

9


**f.** What is the count of unique words starting with the substring "peace"?

In [14]:
# regex101: /^peace/
peace_prefix = {w for w in unique_words if w.startswith("peace")}
print("{:,}".format(len(peace_prefix)))

9


**g.** What is the count of unique words ending with the substring "peace"?

In [15]:
# regex101: /peace$/
peace_suffix = {w for w in unique_words if w.endswith("peace")}
print("{:,}".format(len(peace_suffix)))

1


**h.** What is the count of unique words containing the substring "peace" but neither start nor end with it?

In [16]:
# regex101: /peace/  (excluding ^peace and peace$)
peace_middle = {w for w in unique_words if ("peace" in w and not w.startswith("peace") and not w.endswith("peace"))}
print("{:,}".format(len(peace_middle)))

0


**i.** Which is the more common substring in all words "war" or "peace" ?

In [17]:
war_token_hits   = sum(1 for w in word_tokens if "war" in w)
peace_token_hits = sum(1 for w in word_tokens if "peace" in w)

print("war (tokens containing it):", war_token_hits)
print("peace (tokens containing it):", peace_token_hits)
print("More common substring in tokens:", "war" if war_token_hits > peace_token_hits else "peace")

war (tokens containing it): 3430
peace (tokens containing it): 5144
More common substring in tokens: peace


**j.** Which is the more common full word "war" or "peace" ?

In [18]:
war_token_count = sum(1 for w in word_tokens if w == "war")
peace_token_count = sum(1 for w in word_tokens if w == "peace")

more_common_word = "war" if war_token_count > peace_token_count else "peace"

print("war (full word tokens): {:,}".format(war_token_count))
print("peace (full word tokens): {:,}".format(peace_token_count))
print("More common full word: {}".format(more_common_word))

war (full word tokens): 516
peace (full word tokens): 3,536
More common full word: peace


## Word frequency histogram

A list of words sorted by frequency, with the 10 most and 10 least frequent in the corpus.


In [19]:
import re
from collections import Counter

token_frequencies = Counter(word_tokens)

# Sorted list of (word, frequency)
# Frequency descending, then word alphabetically for consistent results
words_by_frequency = sorted(token_frequencies.items(), key=lambda item: (-item[1], item[0]))
words_by_rarity = sorted(token_frequencies.items(), key=lambda item: (item[1], item[0]))

top_10 = words_by_frequency[:10]
bottom_10 = words_by_rarity[:10]

print("Top 10 most frequent words:")
for word, freq in top_10:
    print("{}\t{}".format(word, freq))

print("\nBottom 10 least frequent words:")
for word, freq in bottom_10:
    print("{}\t{}".format(word, freq))

Top 10 most frequent words:
the	272618
of	176015
and	138295
to	101670
in	67440
on	36025
for	32558
a	24783
that	24072
its	21289

Bottom 10 least frequent words:
abandon	1
abatement	1
abductees	1
abdullah	1
aberrations	1
abided	1
abiotic	1
abject	1
abkhazian	1
absorbing	1


## Breaking a Caesar cipher

A cipher-breaking exercise. `doc.crypt.txt` holds an English document encrypted with a Caesar cipher, and the English corpus above provides the reference letter frequencies needed to find the shift.


**a.** Use frequency analysis to recover the shift, using the English corpus above as the reference for letter frequencies.

In [20]:
from collections import Counter
import string
import gzip

eng_txt_path = "uncorpus.eng.txt.gz"
crypt_txt_path = "doc.crypt.txt"

def letters_only(text):
    return [ch.lower() for ch in text if ch.isalpha()]

def normalize_letter_counts(letter_counts):
    total = sum(letter_counts.values())
    if total == 0:
        return {ch: 0.0 for ch in string.ascii_lowercase}
    return {ch: letter_counts.get(ch, 0) / total for ch in string.ascii_lowercase}

def caesar_decrypt(text, shift):
    decrypted_chars = []
    for ch in text:
        if "A" <= ch <= "Z":
            base = ord("A")
            decrypted_chars.append(chr((ord(ch) - base - shift) % 26 + base))
        elif "a" <= ch <= "z":
            base = ord("a")
            decrypted_chars.append(chr((ord(ch) - base - shift) % 26 + base))
        else:
            decrypted_chars.append(ch)
    return "".join(decrypted_chars)

def chi_square_score(observed_dist, reference_dist):
    # Lower score means "closer" to the reference language distribution
    score = 0.0
    for ch in string.ascii_lowercase:
        expected = reference_dist.get(ch, 0.0)
        observed = observed_dist.get(ch, 0.0)
        if expected > 0:
            score += ((observed - expected) ** 2) / expected
    return score

with gzip.open(eng_txt_path, "rt", encoding="utf-8", errors="ignore") as eng_file:
    reference_text = eng_file.read()

# Baseline English letter distribution from the provided corpus
reference_letter_dist = normalize_letter_counts(Counter(letters_only(reference_text)))

with open(crypt_txt_path, "r", encoding="utf-8", errors="ignore") as crypt_file:
    encrypted_text = crypt_file.read()

best_shift = None
best_score = None

# Try all 26 shifts and keep the closest match
for shift in range(26):
    candidate_text = caesar_decrypt(encrypted_text, shift)
    candidate_letter_dist = normalize_letter_counts(Counter(letters_only(candidate_text)))
    score = chi_square_score(candidate_letter_dist, reference_letter_dist)

    if best_score is None or score < best_score:
        best_score = score
        best_shift = shift

decoded_text = caesar_decrypt(encrypted_text, best_shift)

print("Best shift: {}".format(best_shift))
print("Score: {}".format(best_score))

Best shift: 8
Score: 0.026570130260955137


**b.** What are the first 10 lines of the decoded message?

In [21]:
decoded_lines = decoded_text.splitlines()

# Quick peek or view to confirm the shift produced readable English
for line in decoded_lines[:10]:
    print(line)

THE UNIVERSAL DECLARATION OF HUMAN RIGHTS

PREAMBLE
WHEREAS RECOGNITION OF THE INHERENT DIGNITY AND OF THE EQUAL AND INALIENABLE RIGHTS OF ALL MEMBERS OF THE HUMAN FAMILY IS THE FOUNDATION OF FREEDOM, JUSTICE AND PEACE IN THE WORLD,

WHEREAS DISREGARD AND CONTEMPT FOR HUMAN RIGHTS HAVE RESULTED IN BARBAROUS ACTS WHICH HAVE OUTRAGED THE CONSCIENCE OF MANKIND, AND THE ADVENT OF A WORLD IN WHICH HUMAN BEINGS SHALL ENJOY FREEDOM OF SPEECH AND BELIEF AND FREEDOM FROM FEAR AND WANT HAS BEEN PROCLAIMED AS THE HIGHEST ASPIRATION OF THE COMMON PEOPLE,

WHEREAS IT IS ESSENTIAL, IF MAN IS NOT TO BE COMPELLED TO HAVE RECOURSE, AS A LAST RESORT, TO REBELLION AGAINST TYRANNY AND OPPRESSION, THAT HUMAN RIGHTS SHOULD BE PROTECTED BY THE RULE OF LAW,

WHEREAS IT IS ESSENTIAL TO PROMOTE THE DEVELOPMENT OF FRIENDLY RELATIONS BETWEEN NATIONS,
